### Create work-funder associations from DataCite metadata

Extracts the `funders` array from **DataCite** works in `locations_mapped` (parsed from `attributes.fundingReferences` by `notebooks/ingest/DataCite.py`), resolves each funder to `mid.funder` by **DOI first, with name fallback**, and writes a junction table analogous to `crossref_work_funders` / `pubmed_work_funders` / `openalex.mid.work_funder`.

Why this exists: many DataCite-registered outputs (Zenodo / Dryad / Figshare deposits, institutional repository records, funder-self-published grants) carry `fundingReferences` that we previously dropped on the floor. The upstream DataCite ingest already extracts them into `locations_mapped(provenance='datacite').funders` (see `notebooks/ingest/DataCite.py` lines ~294-311); this notebook is the missing downstream consumer.

**Bulk-publisher exclusion (oxjobs #478):** three Japanese-origin entries dominate the raw distribution and would skew OpenAlex's funder metadata if ingested as-is:
- *National Institute for Fusion Science* (NIFS) — 38.97M linkages
- *National Institutes of Natural Science* (NINS) — 38.97M (paired with NIFS on every record)
- *Japan Society for the Promotion of Science (JSPS)* — 31.36M (real funder, but bulk-tagged by an upstream JP repository)

Together ~109M of ~115M total raw linkages. After exclusion the next-biggest entry is the European Commission at 659K. Excluded by exact funder_name match in the `exploded` CTE.

**Matching strategy:** DOI-preferred, name-fallback. Probe results showed ~95% of the top non-excluded funders carry empty `funder_doi` on the DataCite side (only EC, NHLBI, a few Crossref Funder ID-bearing entries have DOIs). DOI-only matching would drop most of the signal. The name match uses `lowercase{display_name ∪ alternate_titles}` per oxjob #268.

This table feeds into:
- **WorkAwards** (`datacite_work_funder_awards` leg at priority 4, peer of crossref text-mined)
- **openalex_awards_raw** (provenance `datacite_work_funders`, priority 2)
- **CreateWorksEnriched** (`from_work_awards` leg, oxjobs #371 — auto-flows funders into `work.funders` with no per-feed change)

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.datacite_work_funders
USING delta
AS
WITH exploded AS (
  SELECT
    lm.work_id,
    f.doi  AS funder_doi,
    f.name AS funder_name,
    f.awards AS award_ids
  FROM openalex.works.locations_mapped lm
  LATERAL VIEW EXPLODE(funders) AS f
  WHERE lm.provenance = 'datacite'
    AND lm.work_id IS NOT NULL
    AND (f.doi IS NOT NULL OR f.name IS NOT NULL)
    -- Bulk-publisher distortion guard (oxjobs #478). NIFS + NINS pair (78M
    -- linkages, identical counts, paired on every record) + JSPS (31M, bulk-
    -- tagged by an upstream JP repo). Together ~109M of ~115M total. After
    -- exclusion the next-biggest entry is EC at 659K.
    AND COALESCE(f.name, '') NOT IN (
      'National Institute for Fusion Science',
      'National Institutes of Natural Science',
      'Japan Society for the Promotion of Science (JSPS)'
    )
),
-- DOI matching is preferred when DataCite carries a funder DOI (Crossref
-- Funder IDs — EC, NHLBI, etc.).
matched_by_doi AS (
  SELECT e.work_id, cf.funder_id, e.award_ids
  FROM exploded e
  JOIN openalex.mid.funder cf ON cf.doi = e.funder_doi
  WHERE e.funder_doi IS NOT NULL AND e.funder_doi <> ''
),
-- Name match required for ~95% of the surviving top distribution where
-- DataCite leaves funder_doi empty. Match lowercase against
-- display_name ∪ alternate_titles (oxjobs #268 alt_titles pattern).
-- openalex.mid.funder.alternate_titles is STRING (serialized JSON), per
-- CreateFundersAPI.ipynb; from_json parses to array<string>.
funder_name_variants AS (
  SELECT cf.funder_id, lower(name_variant) AS name_lower
  FROM openalex.mid.funder cf
  LATERAL VIEW EXPLODE(
    array_union(
      array(cf.display_name),
      coalesce(from_json(cf.alternate_titles, 'array<string>'), array())
    )
  ) AS name_variant
  WHERE cf.display_name IS NOT NULL
),
matched_by_name AS (
  SELECT e.work_id, fnv.funder_id, e.award_ids
  FROM exploded e
  JOIN funder_name_variants fnv ON fnv.name_lower = lower(e.funder_name)
  WHERE e.funder_name IS NOT NULL
    AND (e.funder_doi IS NULL OR e.funder_doi = '')
),
matched AS (
  SELECT * FROM matched_by_doi
  UNION ALL
  SELECT * FROM matched_by_name
),
-- Explode each award_id so we can apply is_usable_award_id per element.
-- Databricks rejects UDF calls inside higher-order functions (FILTER, TRANSFORM),
-- so we can't do FILTER(..., x -> is_usable_award_id(x)). OUTER EXPLODE preserves
-- rows where award_ids is NULL/empty so funder-only links survive even when no
-- specific award survives the filter.
flattened AS (
  SELECT
    work_id,
    funder_id,
    CASE WHEN openalex.common.is_usable_award_id(aid) THEN aid END AS aid
  FROM matched
  LATERAL VIEW OUTER EXPLODE(award_ids) AS aid
)
-- Re-aggregate one row per (work_id, funder_id). COLLECT_LIST ignores NULLs so the
-- CASE WHEN drops junk elements naturally. ARRAY_DISTINCT preserves dedup semantics.
SELECT
  work_id,
  funder_id,
  ARRAY_DISTINCT(COLLECT_LIST(aid)) AS award_ids
FROM flattened
GROUP BY work_id, funder_id

### Sanity checks

In [ ]:
%sql
-- Row counts and uniqueness
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT CONCAT(work_id, ':', funder_id)) AS distinct_keys,
  COUNT(DISTINCT work_id) AS distinct_works,
  COUNT(DISTINCT funder_id) AS distinct_funders,
  COUNT(CASE WHEN SIZE(award_ids) > 0 THEN 1 END) AS rows_with_awards
FROM openalex.awards.datacite_work_funders

In [ ]:
%sql
-- Verify composite key uniqueness (should return 0)
SELECT COUNT(*) AS duplicate_keys
FROM (
  SELECT work_id, funder_id, COUNT(*) AS n
  FROM openalex.awards.datacite_work_funders
  GROUP BY work_id, funder_id
  HAVING n > 1
)

In [ ]:
%sql
-- Bulk-publisher exclusion verification: NIFS / NINS / JSPS must contribute zero rows.
-- Per-funder counts for the three excluded entries, joined back through the OA funder IDs.
SELECT
  cf.funder_id,
  cf.display_name,
  COUNT(*) AS rows_post_filter
FROM openalex.awards.datacite_work_funders dwf
JOIN openalex.mid.funder cf ON cf.funder_id = dwf.funder_id
WHERE cf.display_name IN (
  'National Institute for Fusion Science',
  'National Institutes of Natural Science',
  'Japan Society for the Promotion of Science (JSPS)'
)
GROUP BY cf.funder_id, cf.display_name

In [ ]:
%sql
-- Coverage spot-check by top funders: which entities ended up represented? Useful
-- to confirm the name-fallback path fired for funders like the European Commission
-- and Deutsche Forschungsgemeinschaft that had empty funder_doi in the raw data.
SELECT
  cf.funder_id,
  cf.display_name,
  COUNT(*) AS linkage_count
FROM openalex.awards.datacite_work_funders dwf
JOIN openalex.mid.funder cf ON cf.funder_id = dwf.funder_id
GROUP BY cf.funder_id, cf.display_name
ORDER BY linkage_count DESC
LIMIT 20

### Insert awards into `openalex_awards_raw`

Follows the same pattern as `CreateCrossrefWorkFunders` / `CreatePubmedWorkFunders`: explode `award_ids`, generate xxhash64-based IDs, insert with provenance `datacite_work_funders` and priority 2 (peer of `crossref_work_funders` and `pubmed_work_funders`).

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW datacite_work_funder_awards AS
WITH exploded AS (
  SELECT DISTINCT
    dwf.funder_id,
    LOWER(award_id) AS normalized_award_id,
    award_id AS funder_award_id
  FROM openalex.awards.datacite_work_funders dwf
  LATERAL VIEW EXPLODE(award_ids) AS award_id
  WHERE SIZE(award_ids) > 0
),
funders AS (
  SELECT funder_id, display_name, ror_id, doi
  FROM openalex.mid.funder
)
SELECT
  ABS(XXHASH64(CONCAT(f.funder_id, ':', e.normalized_award_id))) % 9000000000 AS id,
  CAST(NULL AS STRING) AS display_name,
  CAST(NULL AS STRING) AS description,
  f.funder_id,
  e.funder_award_id,
  CAST(NULL AS DOUBLE) AS amount,
  CAST(NULL AS STRING) AS currency,
  STRUCT(
    CONCAT('https://openalex.org/F', f.funder_id) AS id,
    f.display_name,
    f.ror_id,
    f.doi
  ) AS funder,
  CAST(NULL AS STRING) AS funding_type,
  CAST(NULL AS STRING) AS funder_scheme,
  'datacite_work_funders' AS provenance,
  CAST(NULL AS DATE) AS start_date,
  CAST(NULL AS DATE) AS end_date,
  CAST(NULL AS INT) AS start_year,
  CAST(NULL AS INT) AS end_year,
  CAST(NULL AS STRUCT<
    given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
  >) AS lead_investigator,
  CAST(NULL AS STRUCT<
    given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
  >) AS co_lead_investigator,
  CAST(NULL AS ARRAY<STRUCT<
    given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
  >>) AS investigators,
  CAST(NULL AS STRING) AS landing_page_url,
  openalex.common.extract_grant_doi(e.funder_award_id) AS doi,
  CONCAT('https://api.openalex.org/works?filter=awards.id:G', ABS(XXHASH64(CONCAT(f.funder_id, ':', e.normalized_award_id))) % 9000000000) AS works_api_url,
  CURRENT_TIMESTAMP() AS created_date,
  CURRENT_TIMESTAMP() AS updated_date
FROM exploded e
JOIN funders f ON f.funder_id = e.funder_id

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'datacite_work_funders' AND priority = 2

In [ ]:
%sql
INSERT INTO openalex.awards.openalex_awards_raw
SELECT *, 2 AS priority
FROM datacite_work_funder_awards